### **Fase 02: Modelado**
**Objetivo:** Utilizar los datos de PostgreSQL para entrenar un modelo de Machine Learning que prediga si una película será un éxito.

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, roc_auc_score
import pickle
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()

def get_connection():
    return psycopg2.connect(
        user=os.getenv("DB_USER"), 
        password=os.getenv("DB_PASSWORD"), 
        host=os.getenv("DB_HOST"), 
        port=os.getenv("DB_PORT"), 
        dbname=os.getenv("DB_NAME") 
    )


engine = get_connection()


**DESARROLLO DEL MODELO**

In [ ]:
#cargar datos 

QUERY = """
SELECT
    m.movie_id,
    m.budget,
    m.revenue,
    m.runtime,
    m.popularity,
    m.vote_count,
    EXTRACT(YEAR FROM m.release_date) AS release_year,
    m.adult,
    m.video,
    m.title,
    m.overview,
    m.tagline,
    g.name AS genre_name
FROM movies m
LEFT JOIN movie_genres mg ON m.movie_id = mg.movie_id
LEFT JOIN genres g ON mg.genre_id = g.genre_id
WHERE m.budget > 0 AND m.revenue > 0;
"""
df_raw = pd.read_sql(QUERY, engine)


#target
df_raw['success'] = (df_raw['revenue'] > df_raw['budget'] * 2).astype(int)

#features
df_raw['adult'] = df_raw['adult'].astype(int)
df_raw['video'] = df_raw['video'].astype(int)
numeric_features = ['budget', 'runtime', 'popularity', 'vote_count', 'release_year', 'adult', 'video']

df_movies_unique = df_raw.drop_duplicates(subset='movie_id').set_index('movie_id')
X_num = df_movies_unique[numeric_features].fillna(df_movies_unique[numeric_features].median())

#one-hot
genre_ohe = df_raw[['movie_id', 'genre_name']].dropna()
genre_ohe['value'] = 1
genres_df = genre_ohe.pivot_table(
    index='movie_id',
    columns='genre_name',
    values='value',
    aggfunc='max',
    fill_value=0
)
genres_df = genres_df.reindex(df_movies_unique.index, fill_value=0)
genre_cols = genres_df.columns



#texto
df_movies_unique['text_combined'] = (
    df_movies_unique['title'].fillna("") + " " +
    df_movies_unique['overview'].fillna("") + " " +
    df_movies_unique['tagline'].fillna("")
)

tfidf = TfidfVectorizer(max_features=256, stop_words='english') 
X_tfidf = tfidf.fit_transform(df_movies_unique['text_combined']).toarray()


pca = PCA(n_components=64, random_state=42)  
X_text = pca.fit_transform(X_tfidf).astype('float32')



# Dataset final
X = np.hstack([X_num.values, genres_df.values, X_text])
y = df_movies_unique['success'].values



# train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)



# Escalado
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



# Modelo
gb = GradientBoostingClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
gb.fit(X_train_scaled, y_train)



#Comprobación
y_pred = gb.predict(X_test_scaled)
y_prob = gb.predict_proba(X_test_scaled)[:, 1]
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))




# Guardado para FastAPI
package = {
    'numeric_features': numeric_features,
    'genre_columns': list(genre_cols),
    'scaler': scaler,
    'tfidf_vectorizer': tfidf,
    'pca': pca,
    'gb_model': gb
}


with open('movie_model.pkl', 'wb') as f:
    pickle.dump(package, f)


print("Modelo para FastAPI guardado")